# baseline_vomega — 3-모델 앙상블 피날레 (yolo11x + yolo26x + RT-DETR)

구성:
- **yolo11x.pt / yolo26x.pt**: 이미 학습된 best.pt (Drive `outputs/yolo/test_run/weights/`에 미리 넣어둠)
- **RT-DETR**: 이 노트북에서 새로 학습 (같은 dataset.yaml → 클래스 인덱스 일치)

산출물 2개:
- `submission_RTDETR.csv` — 학습한 RT-DETR 단독 예측
- `submission_final.csv` — 세 모델 WBF(Weighted Box Fusion) 최종 결합

**사전 준비:** 런타임 → 런타임 유형 변경 → **GPU (A100 권장)**

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 경로 & 설정 — **이 셀만 수정하세요**

- `DRIVE_PROJECT_DIR`: 구글 드라이브 프로젝트 루트
- yolo11.pt / yolo26.pt는 Drive `outputs/yolo/test_run/weights/`에 있다고 가정 (`DRIVE_WEIGHTS_DIR`)
- RT-DETR 학습 하이퍼파라미터 / 앙상블 WBF 가중치 / 출력 CSV 이름을 여기서 조정

### 산출 파일 (로컬 `/content/project/baseline/` 기준)
| 경로 | 내용 |
| --- | --- |
| `data/yolo/dataset.yaml`, `orig_id_map.json` | YOLO 데이터 정의 / 클래스ID 복원 매핑 (세 모델 공유) |
| `outputs/yolo/test_run/weights/yolo11.pt`, `yolo26.pt` | Drive에서 가져온 학습된 YOLO 가중치 |
| `outputs/yolo/<RTDETR_RUN_NAME>/weights/best.pt` | 새로 학습한 RT-DETR 가중치 |
| `outputs/predictions/submission_RTDETR.csv` | **RT-DETR 단독 제출본** |
| `outputs/predictions/submission_final.csv` | **3-모델 앙상블 최종 제출본** |

In [ ]:
import os

# ─── 여기만 수정 ──────────────────────────────────────────────────
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/베이스라인_코랩버전v1.4'

# 미리 학습해 둔 YOLO 가중치 파일명 (Drive outputs/yolo/test_run/weights/ 안)
YOLO11_NAME = 'yolo11.pt'
YOLO26_NAME = 'yolo26.pt'

# RT-DETR 학습 설정 (A100 80GB 기준)
RTDETR_MODEL    = 'rtdetr-l.pt'   # 더 키우려면 rtdetr-x.pt
RTDETR_EPOCHS   = 150
RTDETR_IMGSZ    = 1024            # ★ 학습 때 쓴 값과 동일하게 유지 (앙상블 추론도 이 값 사용)
RTDETR_BATCH    = 16             # 여유로 24까지, 또는 -1(자동)
RTDETR_RUN_NAME = 'rtdetr'       # outputs/yolo/<이 이름>/

# ── 앙상블 튜닝 노브 (재학습 없이 셀 12만 다시 돌려 비교) ──
# RT-DETR가 약하면(예: 0.87) YOLO 비중을 키운다: '3 3 1', '4 4 1', '2 2 0.5' 등
WBF_WEIGHTS    = '3 3 1'  # [yolo11 yolo26 rtdetr]
WBF_IOU        = 0.55     # 0.5 / 0.6 / 0.65 비교 가능
ENS_CONF       = 0.15     # 낮출수록 후보 박스↑ (0.1, 0.05 시도)
ENS_YOLO_IMGSZ = 1280     # YOLO 추론 해상도 (1536 시도 가능)
ENS_MAX_DET    = 4        # 이미지당 최대 검출 수

# 출력 CSV 이름
RTDETR_SUBMISSION = 'submission_RTDETR.csv'
FINAL_SUBMISSION  = 'submission_final.csv'

# 세션이 죽어도 RT-DETR 이어가기
RESUME        = False   # True면 Drive 백업 last.pt에서 이어서 학습
BACKUP_PERIOD = 10      # N에폭마다 last.pt/best.pt를 Drive로 복사
SAVE_PERIOD   = -1      # N에폭마다 epochN.pt 영구 저장 (-1=끔)
# ────────────────────────────────────────────────────────────────

DRIVE_BASELINE    = os.path.join(DRIVE_PROJECT_DIR, 'baseline')
DRIVE_OUTPUTS     = os.path.join(DRIVE_PROJECT_DIR, 'outputs')
DRIVE_DATA_ZIP    = os.path.join(DRIVE_PROJECT_DIR, 'sprint_ai_project1_data.zip')
DRIVE_WEIGHTS_DIR = os.path.join(DRIVE_OUTPUTS, 'yolo', 'test_run', 'weights')
DRIVE_YOLO11      = os.path.join(DRIVE_WEIGHTS_DIR, YOLO11_NAME)
DRIVE_YOLO26      = os.path.join(DRIVE_WEIGHTS_DIR, YOLO26_NAME)
DRIVE_RTDETR_DIR  = os.path.join(DRIVE_OUTPUTS, 'yolo', RTDETR_RUN_NAME)

LOCAL_PROJECT_DIR = '/content/project'
LOCAL_BASELINE    = os.path.join(LOCAL_PROJECT_DIR, 'baseline')
LOCAL_OUTPUTS     = os.path.join(LOCAL_BASELINE, 'outputs')
LOCAL_DATA_DIR    = os.path.join(LOCAL_PROJECT_DIR, 'sprint_ai_project1_data')

# 앙상블이 쓸 로컬 YOLO 가중치 경로 (cwd=LOCAL_BASELINE 기준 상대경로)
LOCAL_YOLO11 = f'outputs/yolo/test_run/weights/{YOLO11_NAME}'
LOCAL_YOLO26 = f'outputs/yolo/test_run/weights/{YOLO26_NAME}'

print('--- Drive ---')
for p in [DRIVE_PROJECT_DIR, DRIVE_BASELINE, DRIVE_DATA_ZIP, DRIVE_YOLO11, DRIVE_YOLO26]:
    print(f'  exists={os.path.exists(p)}  {p}')
print('--- Run config ---')
print(f'  RT-DETR : {RTDETR_MODEL}  epochs={RTDETR_EPOCHS} imgsz={RTDETR_IMGSZ} batch={RTDETR_BATCH} name={RTDETR_RUN_NAME}')
print(f'  Ensemble: weights=[{WBF_WEIGHTS}] wbf_iou={WBF_IOU} conf={ENS_CONF} yolo_imgsz={ENS_YOLO_IMGSZ} max_det={ENS_MAX_DET}')
print(f'  outputs : {RTDETR_SUBMISSION}, {FINAL_SUBMISSION}')
print(f'  RESUME={RESUME}  BACKUP_PERIOD={BACKUP_PERIOD}  SAVE_PERIOD={SAVE_PERIOD}')

## 3. Drive → Colab 로컬 복제

- baseline: 그대로 복사
- 데이터셋: zip 한 덩어리로 받아 로컬 unzip
- 이미 있으면 skip (재실행 안전)

In [ ]:
import os, shutil, subprocess, time

os.makedirs(LOCAL_PROJECT_DIR, exist_ok=True)

def copy_dir(src, dst, label):
    if not os.path.exists(src):
        print(f'[skip] {label}: Drive에 없음'); return
    if os.path.exists(dst):
        print(f'[skip] {label}: 이미 로컬에 있음'); return
    t0 = time.time()
    shutil.copytree(src, dst)
    print(f'[ok]   {label} 복사 완료 ({time.time()-t0:.1f}s)')

copy_dir(DRIVE_BASELINE, LOCAL_BASELINE, 'baseline/')

if os.path.exists(LOCAL_DATA_DIR):
    print(f'[skip] dataset: 이미 해제됨')
else:
    assert os.path.exists(DRIVE_DATA_ZIP), f'데이터 zip 없음: {DRIVE_DATA_ZIP}'
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    print('[..] dataset 해제 중 (수 분 소요)...')
    t0 = time.time()
    subprocess.run(['unzip', '-q', DRIVE_DATA_ZIP, '-d', LOCAL_DATA_DIR], check=True)
    print(f'[ok] dataset 해제 완료 ({time.time()-t0:.1f}s) → {LOCAL_DATA_DIR}')
    print('     하위:', sorted(os.listdir(LOCAL_DATA_DIR))[:5])

## 4. 작업 디렉토리 + config 경로 갱신

In [ ]:
import os, yaml

os.chdir(LOCAL_BASELINE)
print('CWD:', os.getcwd())

# 복사된 default.yaml이 빈 파일로 올라오는 경우가 있어, 필요한 설정을 셀에서 직접 채운다.
cfg_path = 'configs/default.yaml'
try:
    with open(cfg_path, encoding='utf-8') as f:
        cfg = yaml.safe_load(f) or {}
except FileNotFoundError:
    cfg = {}

cfg.setdefault('data', {})
cfg['data'].update({
    'data_root':         LOCAL_DATA_DIR,
    'train_images':      cfg['data'].get('train_images', 'train_images'),
    'test_images':       cfg['data'].get('test_images', 'test_images'),
    'train_annotations': cfg['data'].get('train_annotations', 'train_annotations'),
    'processed_dir':     './data/processed',
    'val_ratio':         cfg['data'].get('val_ratio', 0.2),
    'num_workers':       2,
})
cfg.setdefault('output', {})
cfg['output'].setdefault('prediction_dir', './outputs/predictions')
cfg['output'].setdefault('checkpoint_dir', './outputs/checkpoints')

os.makedirs('configs', exist_ok=True)
with open(cfg_path, 'w', encoding='utf-8') as f:
    yaml.dump(cfg, f, allow_unicode=True, default_flow_style=False)

print('data_root:', cfg['data']['data_root'])
print('--- default.yaml ---')
print(open(cfg_path, encoding='utf-8').read())

## 5. 패키지 설치 (베이스 + YOLO + 앙상블)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q ultralytics ensemble-boxes

## 6. GPU 확인

In [ ]:
import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 7. 어노테이션 전처리 (최초 1회)

In [ ]:
!python scripts/preprocess.py

## 8. COCO → YOLO 형식 변환 (최초 1회)

이 변환이 만드는 `dataset.yaml` / `orig_id_map.json`을 RT-DETR 학습과 3-모델 앙상블이 **공유**합니다.
(yolo11/yolo26도 같은 변환으로 학습됐기에 클래스 인덱스가 일치)

In [ ]:
!python scripts/convert_to_yolo.py

## 9. 학습된 YOLO 가중치 로컬로 복사

Drive `outputs/yolo/test_run/weights/`의 yolo11.pt, yolo26.pt를 로컬로 가져옵니다 (앙상블에서 사용).

In [ ]:
import os, shutil

for src, rel, name in [(DRIVE_YOLO11, LOCAL_YOLO11, YOLO11_NAME),
                       (DRIVE_YOLO26, LOCAL_YOLO26, YOLO26_NAME)]:
    assert os.path.exists(src), f'Drive에 가중치 없음: {src}'
    os.makedirs(os.path.dirname(rel), exist_ok=True)
    if not os.path.exists(rel):
        shutil.copy2(src, rel)
    print(f'[ok] {name} → {rel}  (exists={os.path.exists(rel)})')

In [ ]:
# ── 9-b. (세션 재시작 시) RT-DETR 가중치를 Drive에서 복원 → 학습(셀 10) 건너뛰기 ──
# 첫 학습 전이면 Drive에 아직 없으니 안내만 하고 넘어감 (셀 10에서 학습).
import os, shutil

local_rtdetr = f'outputs/yolo/{RTDETR_RUN_NAME}/weights/best.pt'
drive_rtdetr = os.path.join(DRIVE_RTDETR_DIR, 'weights', 'best.pt')

if os.path.exists(local_rtdetr):
    print('[skip] 로컬에 RT-DETR best.pt 있음 → 셀 10 건너뛰고 셀 11~12로:', local_rtdetr)
elif os.path.exists(drive_rtdetr):
    os.makedirs(os.path.dirname(local_rtdetr), exist_ok=True)
    shutil.copy2(drive_rtdetr, local_rtdetr)
    print('[ok] Drive에서 RT-DETR 복원 완료 → 셀 10(학습) 건너뛰고 셀 11~12로 진행')
    print('     ', drive_rtdetr, '→', local_rtdetr)
else:
    print('[info] RT-DETR best.pt가 Drive/로컬 어디에도 없음 → 셀 10(학습)을 실행하세요.')

## 10. RT-DETR 학습

`BACKUP_PERIOD`에폭마다 `last.pt`/`best.pt`가 Drive(`DRIVE_RTDETR_DIR`)로 복사됩니다.

**세션이 죽어서 이어서 학습하려면:** 1~9 셀 재실행 → 2번 셀에서 `RESUME=True`, `RTDETR_RUN_NAME` 동일하게 → 이 셀 실행.

In [ ]:
resume_flag = '--resume' if RESUME else ''

!python train_rtdetr.py \
    --model {RTDETR_MODEL} \
    --epochs {RTDETR_EPOCHS} \
    --imgsz {RTDETR_IMGSZ} \
    --batch {RTDETR_BATCH} \
    --patience 50 \
    --output outputs/yolo \
    --name {RTDETR_RUN_NAME} \
    --save_period {SAVE_PERIOD} \
    --backup_dir "{DRIVE_RTDETR_DIR}" \
    --backup_period {BACKUP_PERIOD} \
    {resume_flag}

## 11. RT-DETR 단독 추론 → `submission_RTDETR.csv`

In [ ]:
!python inference_rtdetr.py \
    --checkpoint outputs/yolo/{RTDETR_RUN_NAME}/weights/best.pt \
    --conf 0.15 --iou 0.6 --max_det 4 \
    --out outputs/predictions/{RTDETR_SUBMISSION}

## 12. 3-모델 WBF 앙상블 → `submission_final.csv`

yolo11x(1280+TTA) + yolo26x(1280+TTA) + RT-DETR(1024)을 WBF로 결합한 최종 제출본.

In [ ]:
!python inference_ensemble_vomega.py \
    --yolo11 {LOCAL_YOLO11} \
    --yolo26 {LOCAL_YOLO26} \
    --rtdetr outputs/yolo/{RTDETR_RUN_NAME}/weights/best.pt \
    --weights {WBF_WEIGHTS} \
    --yolo_imgsz {ENS_YOLO_IMGSZ} --rtdetr_imgsz {RTDETR_IMGSZ} \
    --conf {ENS_CONF} --iou 0.6 --wbf_iou {WBF_IOU} --max_det {ENS_MAX_DET} \
    --out outputs/predictions/{FINAL_SUBMISSION}

## 13. 두 제출본 확인

In [ ]:
import os, pandas as pd

for name in [RTDETR_SUBMISSION, FINAL_SUBMISSION]:
    path = f'outputs/predictions/{name}'
    if not os.path.exists(path):
        print(f'[없음] {path}'); continue
    df = pd.read_csv(path)
    print(f'=== {name} ===')
    print(f'  총 예측 수 : {len(df)}')
    print(f'  이미지 수  : {df["image_id"].nunique()}')
    print(f'  클래스 수  : {df["category_id"].nunique()}')
    print(df.head(3).to_string(index=False))
    print()

## 13-b. 제출 바리에이션 5종 + 캐글 설명

아래 코드 셀을 실행하면 `outputs/predictions/`에 **5개 CSV**가 생성됩니다.
각 파일을 캐글에 제출할 때 **Description 칸에 아래 해당 줄을 복사**해 넣으세요.

**공통 베이스** (이미 제출한 0.988): `yolo11x + yolo26x + RT-DETR(l) WBF, TTA on YOLO, max_det=4, weights 2:2:1, wbf_iou 0.55, conf 0.15, yolo_imgsz 1280` — v1~v5는 여기서 **한 가지씩만** 변경.

| 파일명 | 바꾼 것 | 캐글 Description (복사용) |
| --- | --- | --- |
| `submission_v1_w331.csv` | 가중치 3:3:1 | `vomega WBF 3:3:1 (RT-DETR 비중↓) | wbf_iou0.55 conf0.15 yolo1280 +TTA` |
| `submission_v2_w441.csv` | 가중치 4:4:1 | `vomega WBF 4:4:1 (RT-DETR 비중↓↓) | wbf_iou0.55 conf0.15 yolo1280 +TTA` |
| `submission_v3_conf005.csv` | conf 0.05 | `vomega WBF 2:2:1 conf0.05 (저신뢰 포함) | wbf_iou0.55 yolo1280 +TTA` |
| `submission_v4_iou065.csv` | wbf_iou 0.65 | `vomega WBF 2:2:1 wbf_iou0.65 (병합 관대) | conf0.15 yolo1280 +TTA` |
| `submission_v5_img1536.csv` | yolo 1536 | `vomega WBF 2:2:1 yolo_imgsz1536 (고해상도) | wbf_iou0.55 conf0.15 +TTA` |

> 팁: 한 번에 한 변수만 바꿨으니, 베이스(0.988) 대비 점수가 오른 변수를 찾으면 **그 변수들을 조합**해 6번째 제출을 만들 수 있어요 (예: v1이 좋고 v3도 좋으면 3:3:1 + conf0.05).

In [ ]:
# 5가지 제출 바리에이션 생성 (재학습 없음, 각각 다른 설정 → 다른 파일명)
# 공통 베이스 = 이미 제출한 0.988 설정: weights 2:2:1, wbf_iou 0.55, conf 0.15, yolo 1280, max_det 4
import subprocess

BASE = dict(weights='2 2 1', wbf_iou=0.55, conf=0.15, yolo_imgsz=1280, max_det=4)

VARIATIONS = [
    dict(file='submission_v1_w331',    **{**BASE, 'weights': '3 3 1'},
         change='WBF 가중치 2:2:1 → 3:3:1 (RT-DETR 비중↓)'),
    dict(file='submission_v2_w441',    **{**BASE, 'weights': '4 4 1'},
         change='WBF 가중치 2:2:1 → 4:4:1 (RT-DETR 비중 더↓)'),
    dict(file='submission_v3_conf005', **{**BASE, 'conf': 0.05},
         change='conf 0.15 → 0.05 (저신뢰 후보 더 포함)'),
    dict(file='submission_v4_iou065',  **{**BASE, 'wbf_iou': 0.65},
         change='WBF IoU 0.55 → 0.65 (박스 병합 더 관대)'),
    dict(file='submission_v5_img1536', **{**BASE, 'yolo_imgsz': 1536},
         change='YOLO 추론 해상도 1280 → 1536 (고해상도 TTA)'),
]

for v in VARIATIONS:
    out = f"outputs/predictions/{v['file']}.csv"
    print(f"\n===== {v['file']} | {v['change']} =====")
    cmd = (
        f"python inference_ensemble_vomega.py "
        f"--yolo11 {LOCAL_YOLO11} --yolo26 {LOCAL_YOLO26} "
        f"--rtdetr outputs/yolo/{RTDETR_RUN_NAME}/weights/best.pt "
        f"--weights {v['weights']} "
        f"--yolo_imgsz {v['yolo_imgsz']} --rtdetr_imgsz {RTDETR_IMGSZ} "
        f"--conf {v['conf']} --iou 0.6 --wbf_iou {v['wbf_iou']} --max_det {v['max_det']} "
        f"--out {out}"
    )
    subprocess.run(cmd, shell=True, check=True)

print('\n===== 완료: 5개 제출 CSV (outputs/predictions/) =====')
for v in VARIATIONS:
    print(f"  {v['file']}.csv  ←  {v['change']}")

## 14. Drive 백업

`outputs/predictions/`(두 CSV)와 `outputs/yolo/`(RT-DETR 가중치/그래프)를 Drive로 올립니다.

In [ ]:
import shutil, os, time

os.makedirs(DRIVE_OUTPUTS, exist_ok=True)

for folder in ['predictions', 'yolo']:
    src = os.path.join(LOCAL_OUTPUTS, folder)
    dst = os.path.join(DRIVE_OUTPUTS, folder)
    if not os.path.exists(src):
        continue
    t0 = time.time()
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'[ok] {folder} → Drive ({time.time()-t0:.1f}s)')

print('백업 완료:', DRIVE_OUTPUTS)